In [202]:
! pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0.tar.gz
! pip install https://github.com/explosion/spacy-models/releases/download/fr_core_news_sm-3.8.0/fr_core_news_sm-3.8.0.tar.gz
! pip install nltk pandas scikit-learn

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 524.3 kB/s eta 0:00:24
     - ------------------------------------- 0.5/12.8 MB 524.3 kB/s eta 0:00:24
     - ------------------------------------- 0.5/12.8 MB 524.3 kB/s eta 0:00:24
     - ------------------------------------- 0.5/12.8 MB 524.3 kB/s eta 0:00:24
     - ------------------------------------- 0.5/12.8 MB 524.3 kB/s eta 0:00:24
     -- ------------------------------------ 0.8/12.8 MB 376.9 kB/s eta 0:00:32
     -- --------

# Exercise 1 : Lemmatization

In this exercise, the objective is to create your own lemmatizer for french language. We will test different lemmatization approaches : 
* Based on a dictionary
* Based on machine learning approach (you can use sklearn) or define your own architecture with pytorch
* With and without pos tag given as input

In all case you should compare your results and report performances of the proposed algorithm to [spacy](https://spacy.io/models/fr) lemmatizer (the different configuration).

You are free to use any machine-learning algorihtm/model, taking or not the context of sentences such as [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html) or training your own [RNN with pytorch](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html). 
However you must always motivate your choices and compare results of the different configurations.

You will send the report to *thomas.gerald@universite-paris-saclay.fr* in PDF format named as following and the code (notebook with  output of the two exercises in a zip format) :


**report_[firstname]_[lastname].pdf**

The report for the two exercises must not exceed three pages !


## Dataset
To train or build your lemmatizer you have three files in *tabular separated values* format :
* [training-set.tsv](https://thomas-gerald.fr/TMC/resources/data/training-set.tsv) that you can use to train/build your dictionnary/model 
* [testing-set.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-set.tsv) used to evaluate the different approaches
* [testing-gallica.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-gallica.tsv) used as gold standard to evaluate performances [github (in french)](https://github.com/Gallicorpora/Lemmatisation)

In our case we have two possibilities for a lemma:
* (a) A sequence of characters, meaning that "to rule" an "a rule" are the same lemma
* (b) A sequence of characters, meaning that "to rule" represent the verb, a tuple ("rule", "V") while "a rule" is represented by the tuple ("rule", "N") 
In the (a) case the size of the vocabulary (output) will be 
## Spacy :

Below a small example using spacy lemmatization
```python
import spacy
nlp = spacy.load("en_core_web_sm")
text_a = "He is thirty years old"
text_b = "We still are champions"
print(f'Lemmatization A : {[(w.lemma_, w.pos_) for w in nlp(text_a)]}')
print(f'Lemmatization B : {[(w.lemma_, w.pos_) for w in nlp(text_b)]}')
```

### Importations

In [203]:
import spacy
import pandas as pd

### Reading data

You can use pandas to read the data using tabular separator as following

In [204]:
train_file = "data/training-set.tsv"
train_set = pd.read_csv(train_file, sep='\t', names=["token", "lemma", "pos"])

test_file = "data/testing-set.tsv"
test_set = pd.read_csv(test_file, sep='\t', names=["token", "lemma", "pos"])

test_gallica_file = "data/testing-gallica.tsv"
test_gallica_set = pd.read_csv(test_gallica_file, sep='\t', header=0, names=["form","lemma","POS","morph"])

Clean dataset

In [205]:
# Remove empty instances
train_set = train_set.dropna()
test_set = test_set.dropna()
test_gallica_set = test_gallica_set.dropna()

In [206]:
# harmonising the two tagsets 
COARSE = {    # training / testing-set
    "N": "NOUN", "V": "VERB", "A": "ADJ", "ADV": "ADV", "D": "DET",
    "P": "ADP", "C": "CONJ", "PRO": "PRON", "CL": "PRON", "PONCT": "PUNCT",
    "ET": "X", "I": "INTJ",
    "P+D": "ADP", "P+PRO": "ADP",   
}
GALLICA = {  # gallica: the first 3 letters give the category
    "NOM": "NOUN", "VER": "VERB", "ADJ": "ADJ", "ADV": "ADV", "DET": "DET",
    "PRE": "ADP", "CON": "CONJ", "PRO": "PRON", "PON": "PUNCT", "NUM": "NUM",
    "OUT": "X", "RED": "X", "ETR": "X", "INJ": "INTJ", "ABR": "X",
}

def coarse_pos(tag):
    if tag in COARSE:
        return COARSE[tag]
    return GALLICA.get(tag[:3].upper(), tag)

In [207]:
train_set["pos"] = train_set["pos"].apply(coarse_pos)
test_set["pos"] = test_set["pos"].apply(coarse_pos)
test_gallica_set["POS"] = test_gallica_set["POS"].apply(coarse_pos)

In [208]:
test_gallica_set = test_gallica_set.drop(columns=["morph"])

Data exploration

In [209]:
headers = ["Datasets", "Train set", "Test set", "Test Gallica set"]
tag_rows = list(zip(
    train_set["pos"].value_counts().items(),
    test_set["pos"].value_counts().items(),
    test_gallica_set["POS"].value_counts().items()
))[:3]

rows = [
    ["Number of unique tokens/forms", len(train_set['token'].unique()), len(test_set['token'].unique()), len(test_gallica_set['form'].unique())],
    ["Number of unique str lemma", len(train_set['lemma'].unique()), len(test_set['lemma'].unique()), len(test_gallica_set['lemma'].unique())]
]
rows += [["Most common tags", *tags] for tags in tag_rows]

widths = [
    max(len(str(row[col])) for row in [headers] + rows)
    for col in range(len(headers))
]

def format_row(row):
    return "| " + " | ".join(
        str(value).ljust(widths[index])
        for index, value in enumerate(row)
    ) + " |"

print(format_row(headers))
print("|-" + "-|-".join("-" * width for width in widths) + "-|")

for row in rows[:2]:
    print(format_row(row))

i = 0
for tag_train, tag_test, tag_gallica in tag_rows:
    i += 1
    print(format_row([
        "Most common tags",
        tag_train,
        tag_test,
        tag_gallica
    ]))
    if i == 3:
        break

| Datasets                      | Train set       | Test set       | Test Gallica set |
|-------------------------------|-----------------|----------------|------------------|
| Number of unique tokens/forms | 23269           | 4281           | 906              |
| Number of unique str lemma    | 15194           | 3271           | 567              |
| Most common tags              | ('NOUN', 67970) | ('NOUN', 4118) | ('NOUN', 592)    |
| Most common tags              | ('ADP', 42635)  | ('ADP', 2639)  | ('VERB', 495)    |
| Most common tags              | ('DET', 37767)  | ('DET', 2275)  | ('PRON', 357)    |


### Lemmatization - Dictionary based approach


The dictionary is established based on the encountered words in the training set. With this set, we determine the lemma of the testing set.

In [210]:
from collections import Counter, defaultdict

class DictLemmatizerSimple:
    def __init__(self, use_pos=False):
        self.use_pos = use_pos
        self.table = {}

    def _key(self, tok, pos=None):
        return (tok.lower(), pos) if self.use_pos else tok.lower()
    
    def fit(self, rows:pd.DataFrame):
        counts = defaultdict(Counter)
        for _, row in rows.iterrows():
            tok, lem, pos = row
            counts[self._key(tok, pos)][lem.lower()] += 1
        # keep only the most frequent lemma for each entry
        self.table = {k: c.most_common(1)[0][0] for k, c in counts.items()}
        return self

    def predict(self, tok, pos=None):
        return self.table.get(self._key(tok, pos), tok.lower())

In [211]:
dict_a = DictLemmatizerSimple(use_pos=False).fit(train_set)
dict_b = DictLemmatizerSimple(use_pos=True).fit(train_set)
print(f"Entries: (a) {len(dict_a.table)}   (b) {len(dict_b.table)}")

Entries: (a) 21733   (b) 23236


Evaluation metric

We use accuracy:

$$\text{accuracy} = \frac{\text{number of correct predictions}}{\text{total number of tokens}}$$

We report it on three levels:

| level | computed on | what it tells us |
|---|---|---|
| acc | all the tokens of the test set | global performance |
| seen | tokens present in the training set | how well the method uses what it has memorised |
| unseen | tokens absent from the training set (OOV) | whether the method can generalize |

We also report the OOV rate, the proportion of test tokens absent from the training set. It is not a quality measure but a property of the data. It could be usefull to undesrstand the accuracy.

In [212]:
TRAIN_VOCAB = {row["token"].lower() for _, row in train_set.iterrows()}
RESULTS = []

def evaluate(predict, rows, name, show_errors=0):
    # tot: total number of predictions
    # ok: total number of correct prediction
    # tot_iv: total number of predictions present in the train set vocabulary.
    # tot_oov: total number of predictions of unknown words.

    ok = tot = ok_iv = tot_iv = ok_oov = tot_oov = 0
    errors = [] # store up to 50 errors in memory for analyzing where the model fails

    for _, (tok, gold, pos) in rows.iterrows():
        pred = predict(tok, pos)
        good = (pred == gold.lower())
        tot += 1; ok += good
        if tok.lower() in TRAIN_VOCAB:
            tot_iv += 1; ok_iv += good
        else:
            tot_oov += 1; ok_oov += good
        if not good and len(errors) < 50:
            errors.append((tok, pos, gold.lower(), pred))
    res = dict(name=name, 
               acc=ok/tot, # global accuracy
               seen=ok_iv/tot_iv if tot_iv else float("nan"),
               unseen=ok_oov/tot_oov if tot_oov else float("nan"), # success rate for unseen samples. When the model comes across a word it doesn't know, is it still often correct?
               oov=tot_oov/tot) # proportion of unseen samples in the test set. How difficult is my test game?
    if show_errors:
        print(f"\nerrors of {name}:")
        for t, p, g, pr in errors[:show_errors]:
            print(f"  {t:20} ({p:5}) gold={g:20} pred={pr}")
    return res

def report(res, store=RESULTS):
    store.append(res)
    print(f"{res['name']:34} acc={res['acc']:.3f}  seen={res['seen']:.3f}  "
          f"unseen={res['unseen']:.3f}  (OOV {100*res['oov']:.1f}%)")
    return res

identity = lambda tok, pos=None: tok.lower()

In [213]:
report(evaluate(identity, test_set, "baseline (identity)"))
report(evaluate(dict_a.predict, test_set, "dictionary (a) without PoS"))
report(evaluate(dict_b.predict, test_set, "dictionary (b) with PoS"))

baseline (identity)                acc=0.629  seen=0.632  unseen=0.573  (OOV 5.7%)
dictionary (a) without PoS         acc=0.947  seen=0.969  unseen=0.573  (OOV 5.7%)
dictionary (b) with PoS            acc=0.960  seen=0.983  unseen=0.573  (OOV 5.7%)


{'name': 'dictionary (b) with PoS',
 'acc': 0.9598827624106979,
 'seen': 0.983292319647714,
 'unseen': 0.5732620320855615,
 'oov': 0.05709226354033095}

### Dictionary approach — results on testing-set

| method | accuracy | seen | unseen |
|---|---|---|---|
| baseline (lemma = token) | 0.636 | 0.640 | 0.573 |
| dictionary (a) without PoS | 0.948 | 0.970 | 0.573 |
| dictionary (b) with PoS | 0.961 | 0.984 | 0.573 |

OOV rate: 5.6%

The baseline is already 64%, because in French many words are the same as their lemma. So a method is useful only if it goes clearly above this.

On known words the dictionary works very well. It just repeats what it saw in the training set.

The PoS tag adds 1.3 points. 

The important column is unseen: it is 0.573 for the three methods. When a token is not in the table, the dictionary returns the token itself, so it does the same thing as the baseline. On unknown words it is not a lemmatizer at all, and a bigger table would not change this.

The global accuracy is high only because the OOV rate is low here 5.6%, since the test set comes from the same domain as the training set.

Let's evaluate on Gallica, where the vocabulary is different

In [214]:
report(evaluate(identity, test_gallica_set, "baseline — gallica"))
report(evaluate(dict_a.predict, test_gallica_set, "dictionary (a) — gallica"))
report(evaluate(dict_b.predict, test_gallica_set, "dictionary (b) — gallica"))

baseline — gallica                 acc=0.419  seen=0.681  unseen=0.081  (OOV 43.6%)
dictionary (a) — gallica           acc=0.504  seen=0.832  unseen=0.081  (OOV 43.6%)
dictionary (b) — gallica           acc=0.526  seen=0.870  unseen=0.081  (OOV 43.6%)


{'name': 'dictionary (b) — gallica',
 'acc': 0.5258711721224921,
 'seen': 0.8696194635059263,
 'unseen': 0.08077544426494346,
 'oov': 0.4357620556142203}

### Dictionary approach — results on testing-gallica

| method | accuracy | seen | unseen |
|---|---|---|---|
| baseline | 0.419 | 0.681 | 0.081 |
| dictionary (a) | 0.504 | 0.832 | 0.081 |
| dictionary (b) | 0.526 | 0.870 | 0.081 |

OOV rate: 43.6% (against 5.6% on testing-set)

The accuracy of the dictionary drops from 0.961 to 0.526 — same table, different text. The reason is the OOV rate: almost half of the tokens were never seen, because of the old spelling.

On unknown words the score falls to 0.081, against 0.573 on the test set. In old French a word is almost never identical to its lemma, so even the identity fallback stops working. On known words the dictionary still gets 0.870, so what it memorised is still useful — there is just too little of it.

Ultimately, this highlights a major limitation of static dictionary-based approaches when facing cross-domain variations within the same language. Because rules and spelling shift significantly across historical periods, relying solely on a fixed vocabulary is insufficient, pointing to the necessity of more advanced, data-driven NLP methods (such as machine learning or neural models) capable of generalizing morphological rules.

### Improvement Attempt

In the case of unknown word, we search for the closest word: the one with the same prefix. This method was made intuitively, later we will discuss existing methods on the litterature.

In [215]:
from collections import Counter, defaultdict

class DictLemmatizer:
    def __init__(self, use_pos=False):
        self.use_pos = use_pos
        self.table = {}

    def _key(self, tok, pos=None):
        return (tok.lower(), pos) if self.use_pos else tok.lower()
    
    def fit(self, rows:pd.DataFrame):
        counts = defaultdict(Counter)
        for _, row in rows.iterrows():
            tok, lem, pos = row
            counts[self._key(tok, pos)][lem.lower()] += 1
        # keep only the most frequent lemma for each entry
        self.table = {k: c.most_common(1)[0][0] for k, c in counts.items()}
        return self

    def predict(self, tok, pos=None):
        key = self._key(tok, pos)
        if key in self.table:
            return self.table[key]

        token = tok.lower()
        prefix = token[:3]

        candidates = []
        for candidate_key, lemma in self.table.items():
            candidate = candidate_key[0] if self.use_pos else candidate_key

            if candidate.startswith(prefix):
                common_prefix = 0
                for a, b in zip(token, candidate):
                    if a != b:
                        break
                    common_prefix += 1
                candidates.append((common_prefix, candidate, lemma))

        if candidates:
            return max(candidates, key=lambda x: (x[0], -abs(len(x[1]) - len(token))))[2]

        return None

In [216]:
dict_c = DictLemmatizer(use_pos=False).fit(train_set)
dict_d = DictLemmatizer(use_pos=True).fit(train_set)
print(f"entries: (c) {len(dict_c.table)}   (d) {len(dict_d.table)}")

report(evaluate(identity, test_set, "baseline (identity)"))
report(evaluate(dict_c.predict, test_set, "dictionary (c) without PoS"))
report(evaluate(dict_d.predict, test_set, "dictionary (d) with PoS"))

entries: (c) 21733   (d) 23236
baseline (identity)                acc=0.629  seen=0.632  unseen=0.573  (OOV 5.7%)
dictionary (c) without PoS         acc=0.931  seen=0.969  unseen=0.302  (OOV 5.7%)
dictionary (d) with PoS            acc=0.945  seen=0.984  unseen=0.302  (OOV 5.7%)


{'name': 'dictionary (d) with PoS',
 'acc': 0.9446785125480858,
 'seen': 0.9836161119026033,
 'unseen': 0.3016042780748663,
 'oov': 0.05709226354033095}

The introduction of the prefix-matching heuristic for unknown words yields a slight improvement in 'seen' word accuracy, but results in a sharp and problematic drop in 'unseen' accuracy. This indicates that forcing a match based solely on the first three characters often introduces erroneous lemmas rather than generalizing effectively.

In practice, this intuitive approach shows limitation. In French, a lot of words share the same prefix (3 first letters) but has different root and nature. For instance, "contraire" and "contribuer" are totally different. One is a NOUN, the other a VERB. The root are also different: "contrar" (certainly) and the other is "contribuer".